## Data Ingestion

### Document Data Injection for CSV

In [ ]:
# from langchain_core.documents import Document

# from langchain_community.document_loaders import CSVLoader

# loader = CSVLoader(
#     file_path="../data/text_files/njit_orgs.csv",
#     csv_args={'delimiter':',','quotechar':'"','fieldnames':['Index', 'Height', 'Weight']},
#     encoding="utf-8"
# )

# docs = loader.load()
# print(docs[0].page_content[:100])
# print(docs[0].metadata)

Index: id
Height: source_type
Weight: title
None: description,category,tags,metadata_json,url,last_u
{'source': '../data/text_files/njit_orgs.csv', 'row': 0}


### TXT File Directory Injestion

In [ ]:
# import os
# os.makedirs("../data/text_files", exist_ok=True)

In [ ]:
# from langchain_community.document_loaders import DirectoryLoader

# dir_loader = DirectoryLoader(
#     "../data/text_files",
#     glob="**/*.txt",
# )

# documents=dir_loader.load()
# documents

ImportError: cannot import name 'Directory' from 'langchain_community.document_loaders' (c:\Users\Moise\Documents\BoFA-JitHub\react-webapp\backend\rag\.venv\Lib\site-packages\langchain_community\document_loaders\__init__.py)

### Injestion

In [ ]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import CSVLoader
from pathlib import Path

from langchain.embeddings import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings


## Read all PDF inside Directory

In [ ]:

#Read all csv's
def process_all_csv(csv_directory):
    if not csv_directory:
        raise DirectoryNotFoundError(f"CSV directory '{csv_directory}' does not exist.")
    all_docs = []
    csv_dir = Path(csv_directory)

    #Find all CSV files recursively. More useful when utilizing more than 1 file for content
    csv_files = list(csv_dir.glob("**/*.csv"))

    print(f"Found {len(csv_files)} CSV files to process")

    for csv_file in csv_files:
        print(f"\nProcessing: {csv_file.name}")
        try:
            loader = CSVLoader(
                file_path="../data/text_files/njit_orgs.csv",
                encoding="utf-8",
            )
            documents = loader.load()

            #Add source info to metadata

            for doc in documents:
                doc.metadata['source_file'] = csv_file.name
                doc.metadata['file_type'] = 'csv'
                doc.metadata['ingested_at'] = datetime.now().isoformat()
            
            all_docs.extend(documents)
            print(f"[X] Loaded {len(documents)} pages")
        except Exception as e:
            print(f" [O] Error: {e}")
    print(f"\nTotal documents loaded: {len(all_docs)}")
    return all_docs
all_csv_docs = process_all_csv("../data/text_files")

Found 1 CSV files to process

Processing: njit_orgs.csv
[X] Loaded 282 pages

Total documents loaded: 282


## Text Splitting into Chunks ( Chunking )

In [ ]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    ''' Split documents into smaller chunks for better RAG performance '''
    text_splitter = RecursiveCharacterTextSplitter( #Difference between this one and CharacterTextSplitter is that this one will try to split on natural boundaries like paragraphs, sentences, etc.
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #Show example of a chunk

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [37]:
chunks=split_documents(all_csv_docs)
chunks

Split 282 documents into 548 chunks

Example chunk:
Content: id: 294345
source_type: club
title: Splash
description: Splash is an organization that hosts an annual event, also called Splash, where high schoolers can attend classes taught by college students.
ca...
Metadata: {'source': '../data/text_files/njit_orgs.csv', 'row': 0, 'source_file': 'njit_orgs.csv', 'file_type': 'csv'}


[Document(metadata={'source': '../data/text_files/njit_orgs.csv', 'row': 0, 'source_file': 'njit_orgs.csv', 'file_type': 'csv'}, page_content='id: 294345\nsource_type: club\ntitle: Splash\ndescription: Splash is an organization that hosts an annual event, also called Splash, where high schoolers can attend classes taught by college students.\ncategory: Civic Engagement/Advocacy, Departmental\ntags: Civic Engagement/Advocacy, Departmental\nmetadata_json: {"short_name": "Splash", "status": "Active"}\nurl: https://njit.campuslabs.com/engage/organization/splash\nlast_updated: 2026-07-07T19:56:07.107707+00:00'),
 Document(metadata={'source': '../data/text_files/njit_orgs.csv', 'row': 1, 'source_file': 'njit_orgs.csv', 'file_type': 'csv'}, page_content='id: 296172\nsource_type: club\ntitle: BAPS Campus Fellowship\ndescription: BAPS Campus Fellowship strives to better society through individual development by instilling values, promoting spirituality, cultivating skills, and nurturing growth.

## Embedding and vectorStore DB

In [38]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
import uuid


In [39]:
from langchain_core import embeddings
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        ''' Initialize Embeddings Manager '''
        ''' model_name: hugging face model name for sentence embeddings '''
        self.model_name = model_name
        self.model = None
        self._load_model() #Load Model
    def _load_model(self):
        ''' Load Sentance Transformer Model'''
        try:
            print(f"Loading Embedding Model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error while Loading Model {self.model_name}: {e}")
            raise
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        ''' Generate Embeddings for a list of texts '''
        ''' Arguments are a list of text strings to embed '''
        ''' Returns a numpy array of embeddings with shape (len(texts), embedding_dim) '''
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generate embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"generated embeddings with shape: {embeddings.shape}")
        return embeddings
    def get_embedding_dimension(self) -> int:
        ''' Get the embedding dimension of the model '''
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()

##initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading Embedding Model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7348.16it/s]


Model loaded successfully. Embedding dimension: 384


In [ ]:
from langchain_core import vectorstores
###Vector Store

class VectorStore:
    ''' manages document embeddings in a ChromaDB vector store '''

    def __init__(self, collection_name: str = "csv_documents", persist_directory: str = "../data/vector_store"):
        ''' Initialize Vector Store '''
        ''' Arguments: collection_name is the name of the ChromaDB collection '''
        ''' persist_directory is the directory to the vector store '''

        self.collection_name = collection_name
        self.persist_directory = persist_directory 
        self.client = None
        self.collection = None
        self._initialize_store()
    def _initialize_store(self):
        ''' Initialize chromaDB client and collections '''
        try:
            #Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector Store intialized. Collection: {self.collection_name}")
            print(f"Existing documents in eollction: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        ''' Add documents and their embeddings to the vector store '''
        '''Arguments; documents: list of LangChain Documents & embeddings: Corresponding embeddings for the documents '''
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match the number of embeddings")
        print(f"Adding {len(documents)} documents to vector store...")

        #Prepare the data for ChromaDB

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #Document content
            documents_text.append(doc.page_content)

            #Embedding
            embeddings_list.append(embedding.tolist())
        
        # add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector Store intialized. Collection: csv_documents
Existing documents in eollction: 0


In [42]:
###Convert the text to embeddings

texts = [doc.page_content for doc in chunks] 
texts

##Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##Store in the Vector DB

vectorstore.add_documents(chunks, embeddings)

Generate embeddings for 548 texts...


Batches: 100%|██████████| 18/18 [00:11<00:00,  1.62it/s]


generated embeddings with shape: (548, 384)
Adding 548 documents to vector store...
Successfully added 548 documents to vector store
Total documents in collection: 548


## Retriver Pipeline From VectorStore

In [ ]:
class Retriever:
    ''' Handles query-based retrieval from the vector store '''
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.5) -> List[Dict[str, Any]]:
        ''' Retrieve relevant documents from the vector store based on a query '''
        ''' Arguments:
            query: The search query
            top_k: The number of top results to return
            score_threshold: Minimum similarity score for a document to be considered relevant
        '''
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

        #Generate query Embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #Search in the vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )

            #process results

            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    #Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents ( after filtering )")
            else:
                print("No documents found")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = Retriever(vectorstore, embedding_manager)
rag_retriever

In [ ]:
rag_retriever.retrieve("What is the Society of Professional Hispanic Engineers")